# 03_checkpointer_threads

对应 `index.md` 第 3 阶段：`MemorySaver`、`thread_id`、`get_state`、`get_state_history`。

笔记本在 `codes/langgraph/03_checkpointer_threads/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT` 并加载 `codes/langgraph/local.env`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

前两阶段的图都是「一次性」的：每次 `invoke` 从零开始。本阶段给 `compile` 装一个 **checkpointer**，再配一个 **`thread_id`**，图就会把每一步的 state 存下来。于是同一 `thread_id` 的第二次 `invoke` 能接着上次继续；`thread_id` 一换就是另一个会话。第 1 题起都要调 API。

In [1]:
import os
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "03_checkpointer_threads" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langgraph" / "03_checkpointer_threads"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

def load_local_env(env_path: Path) -> None:
    """把 local.env 写入 os.environ（已有键不覆盖）。"""
    if not env_path.is_file():
        return
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("\"'")
        if key and key not in os.environ:
            os.environ[key] = value

ROOT = lab_root()
load_local_env(ROOT.parent / "local.env")
ROOT, bool(os.environ.get("DEEPSEEK_API_KEY"))

(PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langgraph/03_checkpointer_threads'),
 True)

## 1. 装上 MemorySaver，让它记住上一轮

没有 checkpointer 时，每次 `invoke` 都是全新的 state。加上 checkpointer 后，同一 `thread_id` 的多次 `invoke` 会自动接续——**第二次不用再把历史传回去**。

已有 `model`、节点 `agent`、建好的 `graph`（还没 compile），以及字符串 `thread_id`。

1. `import` 一个内存 checkpointer（`langgraph.checkpoint.memory` 下的 `MemorySaver`）
2. 把它传给 `graph.compile(checkpointer=...)`
3. 写配置 `config = {"configurable": {"thread_id": thread_id}}`，`invoke` 的第二个参数就是它
4. 第一次 `invoke`：`{"messages": [HumanMessage("我叫小明。")]}`，带 `config`
5. 第二次 `invoke`：**只传** `{"messages": [HumanMessage("我叫什么？")]}`，仍带同一个 `config`
6. 打印两次返回的**消息条数**，以及第二次最后一条的 `content`

第二次应该能答出名字，且消息条数比第一次多——历史是 checkpointer 补上的，不是你传的。

In [3]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

thread_id = "user-1"

# 作答

from langgraph.checkpoint.memory import MemorySaver

app = graph.compile(checkpointer = MemorySaver())

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

response = app.invoke({"messages": [HumanMessage("我叫小明。")]}, config)
print(len(response["messages"]))
print(response["messages"][-1].content)

response = app.invoke({"messages": [HumanMessage("我叫什么？")]}, config)
print(len(response["messages"]))
print(response["messages"][-1].content)
#评阅
# 对。compile(checkpointer=MemorySaver()) + config 里的 thread_id，
# 第二次只传新消息，返回 2 → 4 条且答出「你叫小明。」——历史确实是 checkpointer 补的。
# 小点：compile(checkpointer = MemorySaver()) 关键词参数两侧不写空格更贴 PEP8。

#参考答案
# from langgraph.checkpoint.memory import MemorySaver
#
# app = graph.compile(checkpointer=MemorySaver())
# config = {"configurable": {"thread_id": thread_id}}
#
# r1 = app.invoke({"messages": [HumanMessage("我叫小明。")]}, config)
# print(len(r1["messages"]))
# r2 = app.invoke({"messages": [HumanMessage("我叫什么？")]}, config)
# print(len(r2["messages"]), r2["messages"][-1].content)

2
你好，小明！很高兴认识你。有什么我可以帮你的吗？
4
你叫小明。


## 2. 换一个 `thread_id` 就是另一个会话

checkpointer 按 `thread_id` 分桶存 state。不同 `thread_id` 之间**互相看不见**。

已有装好 checkpointer 的 `app`，以及两个线程 id：`thread_a = "alice"`、`thread_b = "bob"`。

1. 在 `thread_a` 上告诉它名字：`invoke({"messages": [HumanMessage("我叫小明。")]}, config_a)`
2. 在 `thread_a` 上问：只传 `HumanMessage("我叫什么？")`，拿到第一条回答
3. 在 `thread_b` 上问同样的问题：只传 `HumanMessage("我叫什么？")`，拿到第二条回答
4. 打印两条回答各自所在返回值的**消息条数**和最后一条 `content`

`thread_a` 应该记得，`thread_b` 应该完全不知道——尽管用的是同一个 `app`。

In [4]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

app = graph.compile(checkpointer=MemorySaver())

thread_a = "alice"
thread_b = "bob"

# 作答

configs = {
    name : {
        "configurable": {
            "thread_id": name
        }
    }
    for name in [thread_a, thread_b]
}

app.invoke({"messages": [HumanMessage("我叫小明。")]}, configs[thread_a])

response = app.invoke({"messages": [HumanMessage("我叫什么？")]}, configs[thread_a])
print(len(response["messages"]))
print(response["messages"][-1].content)

response = app.invoke({"messages": [HumanMessage("我叫什么？")]}, configs[thread_b])
print(len(response["messages"]))
print(response["messages"][-1].content)


#评阅
# 对。alice 4 条、记得名字；bob 2 条、完全不知道——同一个 app，靠 thread_id 隔离。
# 用字典推导先生成两个 config 也行，不过这里只有两个固定线程，直接写两个 config 更直白。

#参考答案
# config_a = {"configurable": {"thread_id": thread_a}}
# config_b = {"configurable": {"thread_id": thread_b}}
#
# app.invoke({"messages": [HumanMessage("我叫小明。")]}, config_a)
# ra = app.invoke({"messages": [HumanMessage("我叫什么？")]}, config_a)
# rb = app.invoke({"messages": [HumanMessage("我叫什么？")]}, config_b)
# print(len(ra["messages"]), ra["messages"][-1].content)
# print(len(rb["messages"]), rb["messages"][-1].content)

4
你叫小明。
2
我不知道你叫什么名字呀。你可以告诉我，我该怎么称呼你？


## 3. `get_state`：直接读当前快照

checkpointer 存下的 state，可以用 `app.get_state(config)` 取回来。返回的是一个 **StateSnapshot**，常用三个属性：

- `.values`：当前完整 state（一个 dict，消息在 `values["messages"]`）
- `.next`：接下来要执行的节点名元组；跑完了就是空元组 `()`
- `.config`：这个快照自己的配置（带 `thread_id` / `checkpoint_id`）

已有装好 checkpointer 的 `app` 和 `config`（`thread_id = "user-1"`）。前置代码已经先跑了一轮「我叫小明。」，所以这个线程里有 state。

1. `snap = app.get_state(config)`
2. 打印 `len(snap.values["messages"])`（当前消息条数）
3. 打印 `snap.next`
4. 打印 `snap.values["messages"][-1].content`

In [ ]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

app = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "user-1"}}

app.invoke({"messages": [HumanMessage("我叫小明。")]}, config)

# 作答

snap = app.get_state(config)

print(len(snap.values["messages"]))
print(snap.next)
print(snap.values["messages"][-1].content)
#评阅
# 对。get_state 取回 StateSnapshot：2 条消息、next=()（跑完了）、最后一条就是回复。

#参考答案
# snap = app.get_state(config)
# print(len(snap.values["messages"]))
# print(snap.next)
# print(snap.values["messages"][-1].content)

2
()
你好，小明！很高兴认识你。有什么我可以帮你的吗？


## 4. `get_state_history`：把整条 checkpoint 链列出来

每走一步都会存一个 checkpoint。`app.get_state_history(config)` 返回**从新到旧**的全部快照，是一个可迭代对象，包成 `list` 方便数。

已有装好 checkpointer 的 `app` 和 `config`；前置代码已经跑了两轮对话，线程里有多个 checkpoint。

1. `history = list(app.get_state_history(config))`
2. 打印快照**总数**
3. 逐条打印：序号、该快照的 `snapshot.next`、以及该快照最后一条消息的类型名（没有消息就打印 `None`）

看 `next`：某个快照的 `next` 是 `('agent',)`，说明它停在「agent 即将执行」的位置——这正是后面做中断 / 时光倒流要用的锚点。

In [8]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

app = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "user-1"}}

app.invoke({"messages": [HumanMessage("我叫小明。")]}, config)
app.invoke({"messages": [HumanMessage("我叫什么？")]}, config)

# 作答

history = list(app.get_state_history(config))

print(len(history))
for i, snap in enumerate(history):
    print(i, snap.next, f"{snap.values["messages"][-1].type} | {snap.values["messages"][-1].content}" if snap.values["messages"] else None)
#评阅
# 对。6 个快照从新到旧，next 依次 () 与 ('agent',)；空消息那条也按题面打了 None。
# 两处可留意：
#   1) f-string 里嵌同名双引号 {snap.values["messages"]...} 要 Python 3.12+（PEP 701）才合法。
#      本机 3.14 能跑；想兼容旧版就把内层引号改成单引号，或先取 m = snap.values["messages"]。
#   2) 这里用 m.type 得到的是角色字符串（'human' / 'ai'）；要类名（HumanMessage / AIMessage）得用
#      type(m).__name__。两者都算「类型名」，面试/调试里说清楚是哪一个即可。

#参考答案
# history = list(app.get_state_history(config))
# print(len(history))
# for i, snap in enumerate(history):
#     msgs = snap.values.get("messages") or []
#     last = type(msgs[-1]).__name__ if msgs else None
#     print(i, snap.next, last)

6
0 () ai | 你叫小明。
1 ('agent',) human | 我叫什么？
2 ('__start__',) ai | 你好，小明！有什么我可以帮你的吗？
3 () ai | 你好，小明！有什么我可以帮你的吗？
4 ('agent',) human | 我叫小明。
5 ('__start__',) None


## 5. 对照：没有 checkpointer 会怎样

把同一段两轮对话跑两遍：一遍**不装** checkpointer，一遍装 `MemorySaver`。看消息条数的差别，就明白 checkpointer 到底补上了什么。

已有 `model`、节点 `agent`、建好的 `graph`（还没 compile），以及两个线程 id。

1. `plain = graph.compile()`（不传 checkpointer），用 `thread_plain` 跑两轮：先 `HumanMessage("我叫小明。")`，再**只传** `HumanMessage("我叫什么？")`；打印第二次返回的消息条数
2. `mem = graph.compile(checkpointer=MemorySaver())`，用 `thread_mem` 跑同样两轮；打印第二次返回的消息条数
3. 打印两者最后一条 `content`

无 checkpointer 的那次，第二次只看到新消息（条数少、也答不出名字）；有 checkpointer 的能接上。

In [10]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, MessagesState, StateGraph

model = init_chat_model(
    f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}"
)

def agent(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_edge(START, "agent")
graph.add_edge("agent", END)

thread_plain = "plain-1"
thread_mem = "mem-1"

# 作答

plain = graph.compile()
mem = graph.compile(checkpointer=MemorySaver())

configs = {
    name : {
        "configurable": {
            "thread_id": name
        }
    }
    for name in [thread_plain, thread_mem]
}

plain.invoke({"messages": [HumanMessage("我叫小明。")]}, configs[thread_plain])

response = plain.invoke({"messages": [HumanMessage("我叫什么？")]}, configs[thread_plain])
print(len(response["messages"]))
print(response["messages"][-1].content)

mem.invoke({"messages": [HumanMessage("我叫小明。")]}, configs[thread_mem])

response = mem.invoke({"messages": [HumanMessage("我叫什么？")]}, configs[thread_mem])
print(len(response["messages"]))
print(response["messages"][-1].content)



#评阅
# 对。无 checkpointer：第二次只有 2 条、答不出名字；有 MemorySaver：4 条、答出名字。
# 对照很清楚，checkpointer 的作用一目了然。
# 小点：configs 字典推导对两个固定线程略绕，直接写 config_plain / config_mem 更直白。

#参考答案
# plain = graph.compile()
# mem = graph.compile(checkpointer=MemorySaver())
# cp = {"configurable": {"thread_id": thread_plain}}
# cm = {"configurable": {"thread_id": thread_mem}}
#
# plain.invoke({"messages": [HumanMessage("我叫小明。")]}, cp)
# rp = plain.invoke({"messages": [HumanMessage("我叫什么？")]}, cp)
# print(len(rp["messages"]), rp["messages"][-1].content)
#
# mem.invoke({"messages": [HumanMessage("我叫小明。")]}, cm)
# rm = mem.invoke({"messages": [HumanMessage("我叫什么？")]}, cm)
# print(len(rm["messages"]), rm["messages"][-1].content)

2
抱歉，我不知道你的名字——你还没有告诉我。你可以直接告诉我，我就记住这次对话里的称呼。
4
你叫小明。
